# 🍏 Health & Fitness Agent with Bing Grounding 🍎

Welcome to our **Health & Fitness Agent with Bing Grounding** tutorial! In this notebook, we'll demonstrate how to:

1. **Initialize** a project using Azure AI Foundry.
2. **Create an Agent** with **Bing Grounding** so it can ground answers in real-time web results.
3. **Ask real-world questions** about health and fitness.
4. **Retrieve and display** answers, including URL citations and disclaimers.

### ⚠️ Model Support Note ⚠️
> Grounding with Bing Search runs on Foundry prompt agents and supports current Foundry models, including `gpt-5.4` — the model we use throughout this notebook. Bing grounding executes **server-side** through the Foundry Responses API and returns a grounded answer with inline **URL citations**.

## Prerequisites
- Complete Agent basics notebook - [1-basics.ipynb](1-basics.ipynb)
- A **Grounding with Bing Search** resource connected to your Foundry project. If `BING_PROJECT_CONNECTION_NAME` is not set, this notebook auto-detects the Bing connection in the project. See [Bing grounding tools](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/bing-tools?tabs=prompt-agents&pivots=python) for setup details.

<img src="./seq-diagrams/4-bing-grounding.png" width="30%"/>

- A `.env` file in the parent directory containing:
  ```bash
  PROJECT_ENDPOINT=https://<resource-name>.ai.azure.com/api/projects/<project-name>
  MODEL_DEPLOYMENT_NAME=gpt-5.4
  BING_PROJECT_CONNECTION_NAME=<your-bing-project-connection-name>
  ```

> **Note:** Bing grounding uses the newer, **endpoint-based** `AIProjectClient` and the **Responses API** (`get_openai_client()` → `responses.create`) instead of the connection-string + threads/runs pattern. That's why this notebook uses `PROJECT_ENDPOINT` rather than `PROJECT_CONNECTION_STRING`.

## Let's Explore Bing Grounding!
We'll give our agent Bing grounding so it can gather up-to-date context from the web, then display the **URL citations** for transparency. 🎉


## 1. Initial Setup
We'll load environment variables from `.env`, initialize the endpoint-based **AIProjectClient**, and grab its **OpenAI (Responses API) client** that we'll use to run the agent.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

# Initialize the endpoint-based AIProjectClient and its OpenAI (Responses API) client
try:
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ Successfully initialized AIProjectClient + OpenAI client")
except Exception as e:
    print(f"❌ Error initializing project client: {e}")

## 2. Create a Bing-Grounded Agent 🌐
We'll create a **prompt agent** and attach **BingGroundingTool** so it can ground its answers in live web results. We'll also give it disclaimers about not being a doctor, etc.

Make sure your `MODEL_DEPLOYMENT_NAME` points to a deployed model that supports Bing grounding (for example, `gpt-5.4`).

In [ ]:
from azure.ai.projects.models import (
    PromptAgentDefinition,
    BingGroundingTool,
    BingGroundingSearchToolParameters,
    BingGroundingSearchConfiguration,
    )

BING_CONNECTION_NAME = os.environ.get("BING_PROJECT_CONNECTION_NAME")

def _resolve_bing_connection():
    """Resolve the Bing project connection from config or by auto-detecting it."""
    if BING_CONNECTION_NAME:
        return project_client.connections.get(BING_CONNECTION_NAME)

    bing_connections = [
        connection
        for connection in project_client.connections.list()
        if "bing" in str(getattr(connection, "type", "")).lower()
        or "bing" in str(getattr(connection, "name", "")).lower()
    ]
    if bing_connections:
        return bing_connections[0]

    raise ValueError(
        "No Grounding with Bing Search connection was found in this project. "
        "Set BING_PROJECT_CONNECTION_NAME in .env or create a Bing project connection first.",
    )

def create_bing_grounding_agent():
    """Create a prompt agent that can search the web with Bing grounding."""
    try:
        bing_connection = _resolve_bing_connection()
        agent = project_client.agents.create_version(
            agent_name="health-bing-grounding-agent",
            definition=PromptAgentDefinition(
                model=os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4"),
                instructions="""
                    You are a health and fitness assistant with Bing grounding capabilities.
                    Always:
                    1. Provide disclaimers that you are not a medical professional.
                    2. Encourage professional consultation.
                    3. Use Bing grounding for up-to-date, real-world references.
                    4. Provide brief, helpful answers and cite your sources.
                """,
                tools=[
                    BingGroundingTool(
                        bing_grounding=BingGroundingSearchToolParameters(
                            search_configurations=[
                                BingGroundingSearchConfiguration(
                                    project_connection_id=bing_connection.id,
                                    market="en-US",
                                    count=7,
                                    freshness="Week",
                                    set_lang="en",
                                )
                            ]
                        )
                    )
                ],
            ),
            description="Health & fitness agent grounded with Bing Search.",
        )
        print(f"🎉 Created Bing-grounded agent '{agent.name}', version: {agent.version}")
        return agent
    except Exception as e:
        print(f"❌ Error creating Bing-grounded agent: {e}")
        return None

# Create our Bing-grounded agent
web_agent = create_bing_grounding_agent()

## 3. Asking Questions 💬
Instead of threads/runs, Bing grounding runs through the **Responses API**. For each question we call `openai_client.responses.create(...)`, referencing our agent so it decides when to ground the answer with web context. We store all `(query, response)` pairs so we can review them in the next step.

In [ ]:
web_responses = []

def ask_web_question(agent, user_query):
    """Send a query to the Bing-grounded agent via the Responses API."""
    try:
        print(f"📨 Query: '{user_query}'")
        response = openai_client.responses.create(
            input=user_query,
            tool_choice="required",  # force the agent to use its grounding tool
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )
        print(f"🤖 Response status: {response.status}\n")
        return user_query, response
    except Exception as e:
        print(f"❌ Error asking web question: {e}")
        return user_query, None

if web_agent:
    # We'll ask a few fun questions!
    questions = [
        "What are some new HIIT workout trends I should know about?",
        "What's the current WHO recommendation for sugar intake?",
        "Any news on intermittent fasting for weight management?"
    ]

    for q in questions:
        query, resp = ask_web_question(web_agent, q)
        if resp:
            web_responses.append((query, resp))

## 4. Viewing Answers & URL Citations
For each response we print the agent's grounded answer (`response.output_text`) and the **URL citations** it returned (so you comply with the requirement to show where the data came from). The Responses API attaches citations as `url_citation` annotations on the message content, and grounding/tool calls appear in the response output items.

In [ ]:
def view_web_response(user_query, response):
    """Print the grounded answer plus URL citations from the response annotations."""
    try:
        print("\n🗣️ Q:", user_query)
        print("ASSISTANT:", response.output_text, "\n")

        citations = []
        search_queries = []
        for item in (response.output or []):
            itype = getattr(item, "type", "")
            if itype == "message":
                for c in (getattr(item, "content", None) or []):
                    for ann in (getattr(c, "annotations", None) or []):
                        if getattr(ann, "type", "") == "url_citation":
                            citations.append((getattr(ann, "title", "") or "", ann.url))
            elif "bing" in itype or "web_search" in itype:
                action = getattr(item, "action", None)
                query = getattr(action, "query", None) if action else None
                if query:
                    search_queries.append(query)

        if search_queries:
            print("🔎 Search queries performed:")
            for q in search_queries:
                print(f"    {q}")
        if citations:
            print("🔗 Citations:")
            for title, url in citations:
                print(f"    [{title}] {url}")
        else:
            print("No URL citations returned for this query.")
    except Exception as e:
        print(f"❌ Error viewing web response: {e}")

# Display all queries and agent responses
if web_responses:
    for (query, resp) in web_responses:
        view_web_response(query, resp)

## 5. Cleanup & Best Practices
This notebook **deletes the agent version** at the end to free up resources. In production, you might keep it around for repeated usage.

### Best Practices
1. **Accuracy** – Web results may include partial or conflicting info. Encourage verification with credible sources.
2. **Citation Display** – Show the **URL citations** returned by the Bing grounding tool (shown above) alongside the agent's answer to satisfy its use and display requirements.
3. **Limits** – Keep an eye on usage, rate limits, or policy constraints for grounding requests.
4. **Privacy** – Treat web results as untrusted input, and avoid sending sensitive data in prompts.
5. **Evaluations** – Use `azure-ai-evaluation` for iterative improvement.

In [ ]:
def cleanup_web_agent(agent):
    if agent:
        try:
            project_client.agents.delete_version(
                agent_name=agent.name,
                agent_version=agent.version,
            )
            print(f"🗑️ Deleted Bing-grounded agent: {agent.name} (v{agent.version})")
        except Exception as e:
            print(f"❌ Error cleaning up agent: {e}")
    else:
        print("No agent to clean up.")

# Delete the agent now to free up resources
cleanup_web_agent(web_agent)

# Congratulations! 🎉
You've built a **Bing-Grounded Health & Fitness Agent** that can:
1. **Search** the web via Bing grounding.
2. **Answer** health/fitness questions with disclaimers.
3. **Include** URL citations for the sources it used.

Feel free to expand this approach by combining Bing grounding with other tools (e.g., **FileSearchTool**, **CodeInterpreterTool**) to build a robust advisor.

#### Let's proceed to [5-agents-aisearch.ipynb](./5-agents-aisearch.ipynb)